In [24]:
import pandas as pd
import numpy as np
from collections import deque
import random

def backtest_ticks(
    csv_path,
    init_usdt=1000.0,
    init_tmn=100_000_000.0,
    fee=0.001,
    K=2.0,
    ROLLING=True,
    ROLL_N=20,
    STATIC_LOWER=-100.0,
    STATIC_UPPER=100.0,
    ORDER_QTY_USDT=100.0,
    mode="taker",       # "taker" | "mid" | "maker"
    p_fill=1.0,         # fill probability for mid/maker orders
    tick_size=1.0       # one tick (e.g. 1 TMN)
):
    """
    Backtest USDT↔TMN arbitrage strategy on historical tick data.

    Strategy logic:
    1. Input tick data includes y_bid, y_ask, y_mid (Wallex) and x_mid (Nobitex).
       Spread variable is defined as:
           z = y_mid - x_mid
    2. Rolling bands are calculated:
           upper = mean(z) + K * std(z)
           lower = mean(z) - K * std(z)
       If rolling mode is disabled or not enough samples, static bands are used.
    3. Trade signals:
       - If z > upper → SELL USDT (receive TMN)
       - If z < lower → BUY USDT (pay TMN)
    4. Trade execution modes:
       - "taker": always executed at best bid/ask (100% fill)
       - "mid": order placed at (bid+ask)/2 with fill probability = p_fill
       - "maker": order placed one tick better than best bid/ask
                  (buy at bid+tick, sell at ask-tick) with fill probability = p_fill
    5. Fees:
       - Always applied for any mode.
       - SELL: received TMN = qty * px * (1 - fee)
       - BUY:  paid TMN = qty * px * (1 + fee)
    6. Portfolio tracking:
       - Balances of USDT and TMN are updated after each fill.
       - If insufficient balance, order is skipped.
    7. Benchmark:
       - All portfolio converted into USDT at the first tick
       - Final value compared with strategy performance
       - Report includes both strategy and benchmark results

    Returns:
        trades_df (pd.DataFrame): log of all executed trades
        report (dict): summary performance metrics
    """

    # Load dataset
    df = pd.read_csv(csv_path)

    # Portfolio initial values
    first_mid = df.iloc[0]["y_mid"]
    last_mid  = df.iloc[-1]["y_mid"]

    usdt = init_usdt
    tmn = init_tmn

    zs = deque(maxlen=ROLL_N if ROLLING else 1)
    trades = []

    for i, row in df.iterrows():
        ts = row["ts_utc"]
        w_bid, w_ask, w_mid = row["y_bid"], row["y_ask"], row["y_mid"]
        x_mid = row["x_mid"]
        z = w_mid - x_mid

        # --- Calculate trading bands
        if ROLLING:
            zs.append(z)

        have_roll = (ROLLING and len(zs) == ROLL_N)
        if have_roll:
            mean = np.mean(zs)
            std = np.std(zs, ddof=1) if len(zs) > 1 else 0.0
            upper, lower = mean + K * std, mean - K * std
            mode_band = "rolling"
        else:
            upper, lower = STATIC_UPPER, STATIC_LOWER
            mode_band = "static"

        # ============================
        #   SELL SIGNAL
        # ============================
        if z > upper:
            if usdt >= ORDER_QTY_USDT:
                qty = ORDER_QTY_USDT

                # Select execution price
                if mode == "taker":
                    px = w_bid
                elif mode == "mid":
                    px = (w_bid + w_ask) / 2
                elif mode == "maker":
                    px = w_ask - tick_size
                else:
                    raise ValueError("Invalid mode")

                # Check fill probability
                if mode == "taker" or random.random() < p_fill:
                    tmn_gain = qty * px * (1 - fee)
                    usdt -= qty
                    tmn += tmn_gain
                    trades.append({
                        "ts": ts, "side": "SELL", "price": px, "qty": qty,
                        "z": z, "usdt": usdt, "tmn": tmn, "band_mode": mode_band,
                        "exec_mode": mode
                    })

        # ============================
        #   BUY SIGNAL
        # ============================
        elif z < lower:
            qty = ORDER_QTY_USDT

            if mode == "taker":
                px = w_ask
            elif mode == "mid":
                px = (w_bid + w_ask) / 2
            elif mode == "maker":
                px = w_bid + tick_size
            else:
                raise ValueError("Invalid mode")

            cost = qty * px * (1 + fee)
            if tmn >= cost:
                if mode == "taker" or random.random() < p_fill:
                    tmn -= cost
                    usdt += qty
                    trades.append({
                        "ts": ts, "side": "BUY", "price": px, "qty": qty,
                        "z": z, "usdt": usdt, "tmn": tmn, "band_mode": mode_band,
                        "exec_mode": mode
                    })

    # Convert trades to DataFrame
    trades_df = pd.DataFrame(trades)

    # Final portfolio value
    final_value_strategy = tmn + usdt * last_mid
    init_value = init_tmn + init_usdt * first_mid
    pnl_strategy = final_value_strategy - init_value
    ret_pct_strategy = pnl_strategy / init_value * 100

    # Benchmark: all converted to USDT at start
    init_usdt_benchmark = init_usdt + init_tmn / first_mid
    final_value_benchmark = init_usdt_benchmark * last_mid
    pnl_benchmark = final_value_benchmark - init_value
    ret_pct_benchmark = pnl_benchmark / init_value * 100

    relative = (final_value_strategy / final_value_benchmark - 1) * 100

    report = {
        "init_value": init_value,
        "final_value_strategy": final_value_strategy,
        "pnl_strategy": pnl_strategy,
        "ret_pct_strategy": ret_pct_strategy,
        "final_value_benchmark": final_value_benchmark,
        "pnl_benchmark": pnl_benchmark,
        "ret_pct_benchmark": ret_pct_benchmark,
        "relative_outperformance_pct": relative,
        "trades_count": len(trades_df)
    }

    return trades_df, report


In [30]:
trades_df, report = backtest_ticks(
    csv_path=fr"C:\Users\amirs\OneDrive\Desktop\myAlgoCode\binance bench\USDT\tick\ticks_1.csv",
    init_usdt=1000,
    init_tmn=0,
    fee=0.001,
    K=1.5,
    ROLLING=True,
    ROLL_N=720 * 2 ,
    STATIC_LOWER=-250,
    STATIC_UPPER=250,
    ORDER_QTY_USDT=120,
    mode="maker",     # taker | mid | maker
    p_fill=.2,       # احتمال پر شدن برای maker
    tick_size=1.0     # یک تیک = ۱ تومان
)

print(report)
print(trades_df.head())


{'init_value': 99224500.0, 'final_value_strategy': 112187040.04000078, 'pnl_strategy': 12962540.040000781, 'ret_pct_strategy': 13.06385019828851, 'final_value_benchmark': 112286500.0, 'pnl_benchmark': 13062000.0, 'ret_pct_benchmark': 13.164087498551266, 'relative_outperformance_pct': -0.08857695270511057, 'trades_count': 1155}
                                 ts  side    price  qty      z  usdt  \
0  2025-09-17T13:29:55.325552+00:00  SELL  99099.0  120 -205.5   880   
1  2025-09-17T13:31:03.466330+00:00  SELL  99195.0  120 -180.5   760   
2  2025-09-17T13:31:16.272837+00:00  SELL  99195.0  120 -155.5   640   
3  2025-09-17T13:31:29.402758+00:00  SELL  99195.0  120 -155.5   520   
4  2025-09-17T13:32:16.506102+00:00  SELL  99195.0  120 -198.5   400   

           tmn band_mode exec_mode  
0  11879988.12   rolling     maker  
1  23771484.72   rolling     maker  
2  35662981.32   rolling     maker  
3  47554477.92   rolling     maker  
4  59445974.52   rolling     maker  
